# VinSOC — Kiểm chất lượng SOC-Agent-Traces-100K

Notebook nghiên cứu nguồn bên ngoài, không là nghiệm thu VinSOC. Cần Python và DuckDB 1.5.5; không gọi model hoặc chạy builder. Đọc toàn bộ test (5.031) + validation (4.984). Train đầu 100 chỉ là mẫu viewer chưa pin nên không tải tự động; không suy rộng kết quả mẫu đó ra 89.985 train.

Cell chuẩn bị tải khoảng 24 MB parquet cùng các file metadata/code nhỏ từ revisions cố định và kiểm SHA-256. Nếu đã có files, đặt DATA_DIR tương ứng và bỏ cell tải. Dữ liệu synthetic không được nhập vào benchmark/runtime nghiệm thu hiện tại.


In [ ]:
from pathlib import Path
import hashlib, json, urllib.request
DATA_DIR = Path(".soc-trace-research")
DATA_DIR.mkdir(exist_ok=True)
DATASET_REV = "fe94a95dadbd188c2bf137a9785b82cb5f7865c2"
BUILDER_REV = "d6d3c71465a1bbfa1c8101ee25a647a01e5adb59"
(DATA_DIR/"revision.json").write_text(json.dumps({"sha": DATASET_REV}))
(DATA_DIR/"builder-revision.json").write_text(json.dumps({"sha": BUILDER_REV}))
# Only pinned public files. Never import or execute the downloaded builder.
FILES = {'test.parquet': '9ef309da7ad04f173d89a5f8da238f20db4051b92514531667b30c042da7bd1a', 'validation.parquet': '94cd7987fe3064cd0933d8fabed75d517e955f8faa8723ac07c9b01446cdfcf7', 'README.md': '54e55fce91b4f85c7c0e583a83c52ccdeaad716dca4f9d50375c0f7d122e0873', 'stats.json': '5f9aca71f9566a941d0053395e64cb51e3123be3eed4ae557e54f6cbf05a2e9a', 'policy.py': '89252f5df55c5424e1dfe706f687df1811bc85d036735aeaadd9794341930caa', 'generator.py': 'a7773c13fc5beb478baf517c9088a1ad9b76f5c47f4c3df5c82369a4089a8cce', 'tools_backend.py': '4bd833340a897c280b84b0ae890b70cd6ee9ac4d5b955989b02342d5c6bb6aa9', 'build_dataset.py': 'c6fd85f90fb27282dcd4f42d16df1e2d295a036b9080c68b219700b744a8f31f', 'schema.py': '3c35c58c6419f885ffdd84764fce8795195424b1e3399244b51e573561f1e20d'}
for name, expected in FILES.items():
    path = DATA_DIR/name
    if name.endswith(".parquet"):
        relative = "data/" + name.replace(".parquet", "-00000-of-00001.parquet")
        url = f"https://huggingface.co/datasets/alirezaaminzadeh/soc-agent-traces-100k/resolve/{DATASET_REV}/{relative}"
    elif name in ("README.md", "stats.json"):
        url = f"https://huggingface.co/datasets/alirezaaminzadeh/soc-agent-traces-100k/resolve/{DATASET_REV}/{name}"
    else:
        relative = name if name == "build_dataset.py" else "soc_traces/" + name
        url = f"https://huggingface.co/spaces/alirezaaminzadeh/soc-agent-traces-builder/resolve/{BUILDER_REV}/{relative}"
    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as response:
            data = response.read()
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError("SOURCE_HASH_MISMATCH: " + name)
        path.write_bytes(data)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError("SOURCE_HASH_MISMATCH: " + name)
print("Pinned test/validation and builder sources verified; no model calls.")


In [ ]:
import collections, hashlib, json, re, sys
from pathlib import Path
from datetime import datetime
ROOT=Path(DATA_DIR).resolve()  # DATA_DIR set by the preparation cell below.
import duckdb
con=duckdb.connect(config={'threads':1,'autoload_known_extensions':False,'autoinstall_known_extensions':False})
FIELDS=('alert','environment','ground_truth','trace','tool_calls','evidence')
CONCLUSIONS={'escalate':('This requires immediate escalation.','Escalating to the incident response team.'),'monitor':('monitoring is the proportionate response.','recommend monitoring with a documented rationale.'),'close':('Closing with rationale.','Closing this alert.')}

def walk(obj):
 if isinstance(obj,dict):
  yield obj
  for v in obj.values():yield from walk(v)
 elif isinstance(obj,list):
  for v in obj:yield from walk(v)

def strings(obj):
 if isinstance(obj,str):yield obj
 elif isinstance(obj,dict):
  for v in obj.values():yield from strings(v)
 elif isinstance(obj,list):
  for v in obj:yield from strings(v)

def audit(rows):
 counts=collections.Counter();tools=collections.Counter();arches=collections.Counter();decisions=collections.Counter();ids=set();evidence_ids=set();signatures=set();examples={}
 for row in rows:
  counts['records']+=1;ids.add(row['scenario_id']);arches[row['archetype']]+=1;decisions[row['decision']]+=1
  if row['success']:counts['success_true']+=1
  else:counts['success_false']+=1
  try:parsed={f:json.loads(row[f]) for f in FIELDS}
  except Exception:counts['invalid_json_records']+=1;continue
  trace=parsed['trace'];calls=parsed['tool_calls'];gt=parsed['ground_truth']
  if len(trace)!=row['n_messages'] or len(calls)!=row['n_tool_calls']:counts['count_field_mismatch']+=1
  callmap={}; observed=set();decisive=False;tool_payloads=[]
  try:
   for message in trace:
    if message['role']=='assistant':
     for call in message.get('tool_calls',[]):
      if call['id'] in callmap:counts['duplicate_native_id_within_case']+=1
      callmap[call['id']]=call['function']['name'];json.loads(call['function']['arguments'])
    if message['role']=='tool':
     if callmap.get(message['tool_call_id'])!=message['name']:counts['native_link_mismatch']+=1
     payload=json.loads(message['content']);tool_payloads.append(payload)
     observed.update(v for v in strings(payload) if re.fullmatch(r'EV-\d+',v))
     for obj in walk(payload):
      if 'decisive' in obj:decisive=True
      for key in ('event_id','evidence_id'):
       if isinstance(obj.get(key),str):observed.add(obj[key])
  except Exception:counts['trace_parse_or_structure_failure']+=1
  if decisive:counts['visible_decisive_marker_records']+=1
  try:
   match=re.search(r'```json\s*(.*?)\s*```',trace[-1]['content'],re.S)
   final=json.loads(match.group(1)); cited=set(final['evidence_ids']);evidence_ids.update(cited)
   if (final['verdict']==gt['verdict'])!=row['success']:counts['success_vs_verdict_agreement_mismatch']+=1
   if final['decision']!=row['decision'] or final['confidence']!=row['confidence']:counts['final_flat_field_mismatch']+=1
   missing=sorted(cited-observed)
   if missing:
    counts['cited_ids_not_observed_in_tool_messages']+=1
    counts['unobserved_citation_success_true' if row['success'] else 'unobserved_citation_success_false']+=1
    examples.setdefault('unobserved_citation',{'scenario_id':row['scenario_id'],'missing_ids':missing})
   prose=trace[-1]['content'].split('```json')[0]
   advertised=[decision for decision,strings in CONCLUSIONS.items() if any(s in prose for s in strings)]
   if advertised and advertised!=[final['decision']]:
    counts['conclusion_template_vs_final_decision_mismatch']+=1
    examples.setdefault('conclusion_mismatch',{'scenario_id':row['scenario_id'],'success':row['success'],'prose_decision':advertised,'json_decision':final['decision']})
  except Exception:counts['final_report_parse_failure']+=1
  for call in calls:tools[call['tool']]+=1
  signatures.add((row['archetype'],tuple(c['tool'] for c in calls)))
  outside=[]
  for call,payload in zip(calls,tool_payloads):
   if call['tool']=='get_surrounding_events':
    center=datetime.fromisoformat(parsed['alert']['timestamp']); minutes=call['arguments'].get('window_minutes',30)
    for event in payload.get('events',[]):
     try:
      if abs((datetime.fromisoformat(event['timestamp'])-center).total_seconds())>minutes*60:outside.append(event['event_id'])
     except Exception:counts['timestamp_parse_failure']+=1
  if outside:
   counts['events_outside_plus_minus_requested_minutes_records']+=1
   examples.setdefault('time_window_diagnostic',{'scenario_id':row['scenario_id'],'out_of_window_event_ids':outside})
 for key in ['invalid_json_records','count_field_mismatch','duplicate_native_id_within_case','native_link_mismatch','trace_parse_or_structure_failure','final_report_parse_failure','final_flat_field_mismatch','success_vs_verdict_agreement_mismatch']:
  counts.setdefault(key,0)
 return {'counts':dict(counts),'archetypes':dict(arches),'decisions':dict(decisions),'tools':dict(tools),'unique_scenario_ids':len(ids),'unique_cited_id_strings_across_cases':len(evidence_ids),'archetype_tool_sequence_signature_count':len(signatures),'examples':examples},ids,set(arches),signatures

result={'scope':'offline_external_synthetic_dataset_quality_audit_not_vinsoc_acceptance','dataset_revision':json.loads((ROOT/'revision.json').read_text())['sha'],'builder_revision':json.loads((ROOT/'builder-revision.json').read_text())['sha'],'engine':'duckdb '+duckdb.__version__,'new_model_calls':0,'splits':{},'hashes':{},'limitations':['Only validation and test parquet fully examined; train sample is first100 from unpinned datasets-server viewer, not a complete training audit.','Time-window diagnostic uses +/- requested minutes; builder does not implement exact temporal filtering.','Citation diagnostic checks event/evidence IDs returned in tool messages, not truth of prose/semantic correctness.','Builder revision was inspected, not independently reproduced against dataset revision.']}
sets={}
for split in ['test','validation']:
 p=ROOT/(split+'.parquet');cols=[d[0] for d in con.execute('describe select * from read_parquet(?)',[str(p)]).fetchall()]
 data=con.execute('select * from read_parquet(?)',[str(p)]).fetchall()
 rows=[dict(zip(cols,r)) for r in data]
 stats,ids,arches,sigs=audit(rows);result['splits'][split]=stats;sets[split]=(ids,arches,sigs)
 result['hashes'][p.name]=hashlib.sha256(p.read_bytes()).hexdigest()
train_path=ROOT/'train-first100.json'
if train_path.exists():
 train=[r['row'] for r in json.loads(train_path.read_text())['rows']]
 stats,ids,arches,sigs=audit(train);result['splits']['train_first100_viewer_sample']=stats;sets['train_first100_viewer_sample']=(ids,arches,sigs)
result['cross_split']={'test_validation_scenario_id_overlap':len(sets['test'][0]&sets['validation'][0]),'test_validation_archetype_overlap':len(sets['test'][1]&sets['validation'][1]),'test_validation_archetype_tool_sequence_signature_overlap':len(sets['test'][2]&sets['validation'][2]),'test_signatures_also_in_first100_train':len(sets['test'][2]&sets['train_first100_viewer_sample'][2]) if 'train_first100_viewer_sample' in sets else None}
for name in ['README.md','stats.json','policy.py','generator.py','tools_backend.py','build_dataset.py','schema.py']:
 result['hashes'][name]=hashlib.sha256((ROOT/name).read_bytes()).hexdigest()
(ROOT/'audit_receipt.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:({s:v['counts'] for s,v in val.items()} if k=='splits' else val) for k,val in result.items() if k in ('dataset_revision','builder_revision','splits','cross_split','limitations')},ensure_ascii=False,indent=2))
